In [1]:
import numpy as np
import xarray as xr
import os
import glob
import matplotlib.pyplot as plt
import pandas as pd
import json
import cftime
from cftime import DatetimeNoLeap
from scipy import stats
from scipy.interpolate import interp1d
import dask
from tqdm import tqdm
from mpl_axes_aligner import align
from xmip.preprocessing import rename_cmip6
%matplotlib inline


from utils import QACC_utils
from utils.config import model, min_lat, max_lat, min_year_late_cent, max_year_late_cent

In [2]:
ds_ssp245 = xr.open_dataset('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
            s='ssp245', l=str(min_lat), m=model))
ds_arise = xr.open_dataset('intermediate_outputs/fin/{s}_{l}_{m}.nc'.format(
            s='ARISE', l=str(min_lat), m=model))

In [3]:
ds_ssp245

<xarray.Dataset> Size: 9kB
Dimensions:                 (month: 12, bnds: 2, plev: 19)
Coordinates:
    height                  float64 8B ...
  * month                   (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * plev                    (plev) float64 152B 1e+05 9.25e+04 ... 500.0 100.0
Dimensions without coordinates: bnds
Data variables: (12/32)
    lat_bounds              (month, bnds) float64 192B ...
    lon_bounds              (month, bnds) float64 192B ...
    tas                     (month) float64 96B ...
    number_ens_mems_meaned  (month) float64 96B ...
    rsds                    (month) float64 96B ...
    rsus                    (month) float64 96B ...
    ...                      ...
    Qmoist                  (month) float64 96B ...
    Qdry                    (month) float64 96B ...
    IRF_aer                 (month) float64 96B ...
    IRF_aer_cs              (month) float64 96B ...
    albedo                  (month) float64 96B ...
    CRF                     (month) float64 96B ...

### Radiative kernels
Kernels are from Smith et al., 2018, for HadGEM2: https://archive.researchdata.leeds.ac.uk/940/
Downloaded Sunday 11th February 2024. 

See also Pendergrass' helpful Github with some Matlab code https://github.com/apendergrass/cam5-kernels/blob/master/scripts/kernel_demo.m


In [4]:
kernels17 = xr.open_dataset('data/kernels_HadGem2/HadGEM2_net_TOA_L17.nc') # note that kernels stop at tropopause, hence only 17 levels, but these match the first 17 levels of the UKESM data
kernels38 = xr.open_dataset('data/kernels_HadGem2/HadGEM2_net_TOA_L38.nc') # need both as the surface ones are only included in L38 version

kernels_s = kernels38[['tsurf', 'tsurf_cs', 'albedo', 'albedo_cs']]
kernels = xr.merge([kernels17, kernels_s])


## take an arctic spatial mean:
kernels = kernels.sel(lat=slice(min_lat, max_lat)).mean('lon')
weights = np.cos(np.deg2rad(kernels.lat))
weights.name = "weights"
kernels_w = kernels.weighted(weights)
kernels = kernels_w.mean("lat")
kernels

<xarray.Dataset> Size: 4kB
Dimensions:    (month: 12, plev: 17)
Coordinates:
  * month      (month) float32 48B 1.0 2.0 3.0 4.0 5.0 ... 9.0 10.0 11.0 12.0
  * plev       (plev) float32 68B 1e+03 2e+03 3e+03 ... 8.5e+04 9.25e+04 1e+05
Data variables:
    ta         (month, plev) float32 816B -0.2456 -0.2101 ... -0.1745 -0.125
    ta_cs      (month, plev) float32 816B -0.245 -0.2093 ... -0.1169 -0.06779
    q          (month, plev) float32 816B 0.06915 0.06784 ... -0.02917 -0.00239
    q_cs       (month, plev) float32 816B 0.06972 0.06845 ... -0.02197 0.004028
    tsurf      (month) float32 48B -0.866 -0.8884 -0.8637 ... -0.8509 -0.9099
    tsurf_cs   (month) float32 48B -1.405 -1.412 -1.463 ... -1.528 -1.497 -1.439
    albedo     (month) float32 48B -0.006553 -0.1132 ... -0.01353 -0.0001475
    albedo_cs  (month) float32 48B -0.009024 -0.1584 ... -0.02211 -0.0002035

In [5]:
kernels

<xarray.Dataset> Size: 4kB
Dimensions:    (month: 12, plev: 17)
Coordinates:
  * month      (month) float32 48B 1.0 2.0 3.0 4.0 5.0 ... 9.0 10.0 11.0 12.0
  * plev       (plev) float32 68B 1e+03 2e+03 3e+03 ... 8.5e+04 9.25e+04 1e+05
Data variables:
    ta         (month, plev) float32 816B -0.2456 -0.2101 ... -0.1745 -0.125
    ta_cs      (month, plev) float32 816B -0.245 -0.2093 ... -0.1169 -0.06779
    q          (month, plev) float32 816B 0.06915 0.06784 ... -0.02917 -0.00239
    q_cs       (month, plev) float32 816B 0.06972 0.06845 ... -0.02197 0.004028
    tsurf      (month) float32 48B -0.866 -0.8884 -0.8637 ... -0.8509 -0.9099
    tsurf_cs   (month) float32 48B -1.405 -1.412 -1.463 ... -1.528 -1.497 -1.439
    albedo     (month) float32 48B -0.006553 -0.1132 ... -0.01353 -0.0001475
    albedo_cs  (month) float32 48B -0.009024 -0.1584 ... -0.02211 -0.0002035

In [11]:
delta_albedo_perc = 100*(ds_arise['albedo'] - ds_ssp245['albedo'])/ds_ssp245['albedo']
R_albedo = delta_albedo_perc*kernels['albedo']

